# Week 5 — Colour Palette & Visual Harmony Engine
**Tools:** OpenCV, scikit-learn (KMeans), Matplotlib
**Dataset:** Logo Dataset → colour extraction

In [ ]:
!pip install opencv-python-headless scikit-learn matplotlib numpy -q
import cv2,numpy as np,matplotlib.pyplot as plt,json
from pathlib import Path
from sklearn.cluster import KMeans
print('Libraries loaded ✓')

In [ ]:
def extract_palette(image_path_or_array,n=5):
    if isinstance(image_path_or_array,str):
        img=cv2.cvtColor(cv2.imread(image_path_or_array),cv2.COLOR_BGR2RGB)
    else: img=(image_path_or_array*255).astype(np.uint8)
    pix=cv2.resize(img,(100,100)).reshape(-1,3).astype(np.float32)
    km=KMeans(n_clusters=n,random_state=42,n_init='auto').fit(pix)
    cols=km.cluster_centers_.astype(int)
    prop=np.bincount(km.labels_)/len(km.labels_)
    order=np.argsort(-prop); cols,prop=cols[order],prop[order]
    hex_c=['#{:02x}{:02x}{:02x}'.format(*c) for c in cols]
    return hex_c,cols,prop
print('extract_palette() ready ✓')

In [ ]:
# Colour psychology mapping
COLOR_PSY={
    'red':    {'emotions':['energy','passion','urgency'],'industries':['Food & Beverage','Entertainment']},
    'blue':   {'emotions':['trust','calm','professional'],'industries':['Technology','Finance','Healthcare']},
    'green':  {'emotions':['growth','nature','balance'],'industries':['Sustainability','Health']},
    'yellow': {'emotions':['optimism','warmth','youthful'],'industries':['Education','Food & Beverage']},
    'orange': {'emotions':['fun','affordable','friendly'],'industries':['E-Commerce','Sports']},
    'purple': {'emotions':['luxury','creativity','mystery'],'industries':['Beauty','Fashion']},
    'black':  {'emotions':['elegance','power','premium'],'industries':['Luxury','Technology']},
    'white':  {'emotions':['clean','minimal','pure'],'industries':['Healthcare','Technology']},
}
def classify_color(hex_c):
    r,g,b=int(hex_c[1:3],16),int(hex_c[3:5],16),int(hex_c[5:7],16)
    if max(r,g,b)<50: return 'black'
    if min(r,g,b)>200: return 'white'
    if r>g and r>b: return 'red' if r>150 else 'orange'
    if g>r and g>b: return 'green'
    if b>r and b>g: return 'blue' if b>150 else 'purple'
    if r>180 and g>150 and b<80: return 'yellow'
    return 'neutral'
print('Color classification ready ✓')

In [ ]:
def visualise_palette(hex_colors,title='Brand Palette'):
    fig,axes=plt.subplots(1,len(hex_colors),figsize=(len(hex_colors)*2,3))
    if len(hex_colors)==1: axes=[axes]
    for ax,hx in zip(axes,hex_colors):
        r,g,b=int(hx[1:3],16),int(hx[3:5],16),int(hx[5:7],16)
        ax.imshow([[[r/255,g/255,b/255]]]); ax.set_title(hx,fontsize=8,fontweight='bold'); ax.axis('off')
        cat=classify_color(hx)
        if cat in COLOR_PSY: ax.set_xlabel(COLOR_PSY[cat]['emotions'][0],fontsize=7,color='gray')
    plt.suptitle(title,fontweight='bold'); plt.tight_layout(); plt.savefig('week5_palette.png',dpi=150); plt.show()
# NovaTech brand palette
novatech=['#1A1A1A','#F9F9F7','#0D1B3E','#5055D8','#C0C0C8']
visualise_palette(novatech,'NovaTech Brand Palette')

In [ ]:
# Industry colour recommendations
INDUSTRY_COLOURS={'Technology':['#0D1B3E','#5055D8','#E8F0FF','#C0C0C8'],'Healthcare':['#004E89','#00B4D8','#E0F7FA'],'Fashion & Apparel':['#1A1A1A','#C9A84C','#FFF8EC'],'Finance':['#003366','#1F8A70','#E8F5F0'],'Sustainability':['#1B4332','#52B788','#D8F3DC']}
print('Industry colour map saved')
with open('colour_config.json','w') as f: json.dump({'psychology':COLOR_PSY,'industry_palettes':INDUSTRY_COLOURS},f,indent=2)
print('colour_config.json saved ✓')

## ✅ Week 5 Deliverables
- [ ] KMeans colour extraction function (5 dominant colours per logo)
- [ ] HEX/RGB palette output with proportions
- [ ] Colour psychology mapping (8 colour categories)
- [ ] Industry-to-palette recommendation table
- [ ] NovaTech palette visualised and saved
- [ ] colour_config.json saved for Streamlit